# 步跡台語 TTS｜1029V1 獨立模型驗證

**操作方式**
1. 在 Colab 選「執行階段 → 變更執行階段類型 → T4 GPU」。
2. 選「執行階段 → 全部執行」。
3. 最後一格有播放器，直接聽新的台語句子，不必下載 WAV。

首次執行需要網路與 GPU，將下載約 1.11 GB 的 KaedeTai 權重，加上約 1 GB 的基礎模型。這是 **未經 Colab GPU 實跑驗證** 的推論測試筆記本，不是已保證成功的模型。
不訓練、不讀取私人 Drive、不修改步跡 V30、不接觸簽章或 RAW199。

來源：
- KaedeTai 原始碼及模型：https://github.com/KaedeTai/GPT-SoVITS ／ https://huggingface.co/KaedeTai/gpt-sovits-tw
- 使用的第三方推論方法：https://github.com/wutiger555/picklemaster-tw/commit/238971ec9dbe40daff4b8657c784c1c15dcc54fd


In [ ]:
# 1. 確認 NVIDIA GPU
import sys, torch, shutil
print("Python:",sys.version.split()[0],"Torch:",torch.__version__)
assert torch.cuda.is_available(), "請先在「執行階段」選 T4 GPU，再全部執行"
assert shutil.which("ffmpeg"), "這個執行階段找不到 ffmpeg"
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 2. 安裝公開原始碼及依賴
from pathlib import Path
import subprocess,sys
BASE=Path("/content/stride_taigi_1029v1");BASE.mkdir(exist_ok=True)
REPO=BASE/"GPT-SoVITS"
def run(*cmd,cwd=None):
    print("執行："," ".join(map(str,cmd))[:180],flush=True)
    subprocess.run([str(p) for p in cmd],cwd=cwd,check=True)
if not (REPO/".git").exists():
    run("git","clone","--filter=blob:none","https://github.com/KaedeTai/GPT-SoVITS.git",REPO)
run("git","checkout","--detach","81959852f75c972a0a78364f8ea6b14133a11f4a",cwd=REPO)
run(sys.executable,"-m","pip","install","-q","--upgrade","pip")
run(sys.executable,"-m","pip","install","-q","-r",str(REPO/"requirements.txt"))
run(sys.executable,"-m","pip","install","-q","huggingface_hub[hf_xet]","soundfile")
print("程式安裝完成")


In [ ]:
# 3. 下載 KaedeTai 模型與必要基礎模型，僅放 Colab 暫存空間
from huggingface_hub import hf_hub_download,snapshot_download
MODEL=REPO/"models";MODEL.mkdir(exist_ok=True)
PRE=REPO/"GPT_SoVITS"/"pretrained_models";PRE.mkdir(exist_ok=True,parents=True)
for filename in ("s1_trilingual.ckpt","s2_r4_e15.pth"):
    p=hf_hub_download(repo_id="KaedeTai/gpt-sovits-tw",filename=filename,local_dir=str(MODEL))
    print(filename,round(Path(p).stat().st_size/1e6,1),"MB")
snapshot_download(repo_id="lj1995/GPT-SoVITS",local_dir=str(PRE),allow_patterns=[
    "chinese-roberta-wwm-ext-large/*",
    "chinese-hubert-base/*",
    "sv/pretrained_eres2netv2w24s4ep4.ckpt"
])
for f in ("chinese-roberta-wwm-ext-large/config.json",
          "chinese-hubert-base/config.json",
          "sv/pretrained_eres2netv2w24s4ep4.ckpt"):
    assert (PRE/f).is_file(),f"基礎模型缺少 {f}"
assert (REPO/"tw_samples"/"demo_02_kin_a_jit_thinn_khi.mp3").is_file()
print("模型檔案檢查通過")


In [ ]:
# 4. 使用固定版第三方推論路徑產生新的台語句，並直接在線上試聽
import json,requests,time,wave
from IPython.display import display,Audio,Markdown
SRC="https://raw.githubusercontent.com/wutiger555/picklemaster-tw/238971ec9dbe40daff4b8657c784c1c15dcc54fd/scripts/voice"
RUNDIR=BASE/"runner"/"scripts"/"voice";RUNDIR.mkdir(exist_ok=True,parents=True)
for file in ("generate.py","lines.json"):
    res=requests.get(SRC+"/"+file,timeout=45);res.raise_for_status()
    (RUNDIR/file).write_bytes(res.content)
# 第三方 script 有一次不安全的權重載入，局部改為 PyTorch 限制型載入。
generator=RUNDIR/"generate.py"
code=generator.read_text(encoding="utf-8")
old='torch.load(repo / "models" / "s2_r4_e15.pth", map_location="cpu", weights_only=False)'
assert code.count(old)==1,"第三方程式版本不同，停止執行以避免錯誤修改"
generator.write_text(code.replace(old,old.replace("weights_only=False","weights_only=True")),encoding="utf-8")
# 本句尚待教育部用字與自然口語進一步驗證；只用來證明模型會生成新音檔。
TAIGI_POJ="Kin-á-ji̍t guá kiânn-lōo."
(RUNDIR/"lines.json").write_text(json.dumps({"stride_test":[["今仔日我行路。",TAIGI_POJ]]},ensure_ascii=False),encoding="utf-8")
OUT=BASE/"output";OUT.mkdir(exist_ok=True)
start=time.monotonic()
run(sys.executable,str(generator),"candidates","--out",str(OUT),"--seeds","1","--only","stride_test",cwd=REPO)
seconds=round(time.monotonic()-start,2)
audio=OUT/"stride_test-1.s1.wav"
assert audio.is_file() and audio.stat().st_size>2000,"沒有產出有效的新 WAV"
with wave.open(str(audio),"rb") as w: dur=w.getnframes()/w.getframerate()
assert dur>0.2,"WAV 時長異常"
display(Markdown("### 新台語試聽：今仔日我行路"))
display(Audio(filename=str(audio),autoplay=False))
print("新檔案：",audio,"長度：",round(dur,2),"秒","執行：",seconds,"秒")
print("注意：產生 WAV 不等於音質或 Android 實際驗收通過。")
